In [3]:
import os
from glob import glob
import pandas as pd
import numpy as np

files = sorted(
    glob(os.path.join(os.getcwd(), '../..' , 'data', '2010-*.parquet')) +
    glob(os.path.join(os.getcwd(), '../..' , 'data', '2011-*.parquet')) +
    glob(os.path.join(os.getcwd(), '../..' , 'data', '2012-*.parquet')) +
    # 2013 Data is missing
    glob(os.path.join(os.getcwd(), '../..' , 'data', '2014-*.parquet')) +
    glob(os.path.join(os.getcwd(), '../..' , 'data', '2015-*.parquet')) +
    # 2016 Data is missing
    glob(os.path.join(os.getcwd(), '../..' , 'data', '2017-*.parquet')) +
    glob(os.path.join(os.getcwd(), '../..' , 'data', '2018-*.parquet')) +
    glob(os.path.join(os.getcwd(), '../..' , 'data', '2019-*.parquet')) +
    glob(os.path.join(os.getcwd(), '../..' , 'data', '2020-*.parquet'))
)
len(files), files

(99,
 ['/home/rainybyte/DataspellProjects/allsky-light-pollution/notebooks/exploratory/../../data/2010-08.parquet',
  '/home/rainybyte/DataspellProjects/allsky-light-pollution/notebooks/exploratory/../../data/2010-09.parquet',
  '/home/rainybyte/DataspellProjects/allsky-light-pollution/notebooks/exploratory/../../data/2010-10.parquet',
  '/home/rainybyte/DataspellProjects/allsky-light-pollution/notebooks/exploratory/../../data/2010-11.parquet',
  '/home/rainybyte/DataspellProjects/allsky-light-pollution/notebooks/exploratory/../../data/2010-12.parquet',
  '/home/rainybyte/DataspellProjects/allsky-light-pollution/notebooks/exploratory/../../data/2011-01.parquet',
  '/home/rainybyte/DataspellProjects/allsky-light-pollution/notebooks/exploratory/../../data/2011-02.parquet',
  '/home/rainybyte/DataspellProjects/allsky-light-pollution/notebooks/exploratory/../../data/2011-03.parquet',
  '/home/rainybyte/DataspellProjects/allsky-light-pollution/notebooks/exploratory/../../data/2011-04.parque

In [5]:
from dask import dataframe as dd
df: dd.DataFrame = dd.read_parquet(files)
df.describe

<bound method DataFrame.describe of Dask DataFrame Structure:
                  date    time exposure filename
npartitions=99                                  
                string  string   string   string
                   ...     ...      ...      ...
...                ...     ...      ...      ...
                   ...     ...      ...      ...
                   ...     ...      ...      ...
Dask Name: read_parquet, 1 expression
Expr=ReadParquetFSSpec(dbcf187)>

In [9]:
import dask.dataframe as dd
from allsky.validation import is_valid_record_series

invalid_rows = ~is_valid_record_series(df)
df = df[~invalid_rows]
df = df.assign(
    timestamp=dd.to_datetime(
        df["date"] + " " + df["time"],
        format="%Y/%m/%d %H:%M:%S",
        errors="coerce",
    )
)
df = df.assign(exposure=df["exposure"].astype(float)).compute()
df

,date,time,exposure,filename,timestamp
1,2010/08/11,19:17:09,0.0000,000021139,2010-08-11 19:17:09
2,2010/08/11,19:24:05,0.0000,000021140,2010-08-11 19:24:05
3,2010/08/11,19:25:05,0.0000,000021141,2010-08-11 19:25:05
4,2010/08/11,19:26:05,0.0000,000021142,2010-08-11 19:26:05
5,2010/08/11,19:27:06,0.0000,000021143,2010-08-11 19:27:06
...,...,...,...,...,...
24,2020/12/31,23:54:12,7.7944,001695201,2020-12-31 23:54:12
25,2020/12/31,23:55:17,7.7944,001695202,2020-12-31 23:55:17
26,2020/12/31,23:56:21,7.7944,001695203,2020-12-31 23:56:21
27,2020/12/31,23:57:26,7.7944,001695204,2020-12-31 23:57:26


In [11]:
plot_df = df.dropna(subset=["timestamp", "exposure"]).sort_values("timestamp").copy()
x = plot_df["timestamp"].map(pd.Timestamp.toordinal).to_numpy()
y = plot_df["exposure"].to_numpy()

slope, intercept = np.polyfit(x, y, 1)
slope, intercept

(np.float64(0.003837432840193372), np.float64(-2814.6825579646584))

In [12]:
from allsky.astronomy import (
    get_times_from_dataframe,
    get_moon_phase,
    get_altaz,
    sun, moon, ASTRONOMICAL_TWILIGHT, NEWMOON_LIGHT_FRACTION
)

times = get_times_from_dataframe(df)

In [13]:
sun_alt, _ = get_altaz(sun, times)
moon_alt, _ = get_altaz(moon, times)
moonlit = get_moon_phase(times)

sun_alt, moon_alt, moonlit

(array([ 11.22487166,   9.9290376 ,   9.74253194, ..., -71.58063892,
        -71.65050944, -71.71780069], shape=(1210365,)),
 array([18.90451793, 17.64943829, 17.46790827, ..., 55.95270158,
        56.13788489, 56.32268646], shape=(1210365,)),
 array([0.05205806, 0.05229125, 0.05232509, ..., 0.95285137, 0.95282821,
        0.95280507], shape=(1210365,)))

In [14]:
sundown_condition = sun_alt < ASTRONOMICAL_TWILIGHT
moondown_condition = moon_alt < ASTRONOMICAL_TWILIGHT
newmoon_condition = moonlit < NEWMOON_LIGHT_FRACTION

In [15]:
ideal_df = df[sundown_condition & (moondown_condition | newmoon_condition)]
ideal_df

,date,time,exposure,filename,timestamp
8,2010/08/11,22:38:00,2.7883,000021274,2010-08-11 22:38:00
9,2010/08/11,22:39:17,2.4769,000021275,2010-08-11 22:39:17
10,2010/08/11,22:40:17,2.4769,000021276,2010-08-11 22:40:17
11,2010/08/11,22:41:34,2.4769,000021277,2010-08-11 22:41:34
12,2010/08/11,22:42:34,2.4769,000021278,2010-08-11 22:42:34
...,...,...,...,...,...
1,2020/12/25,05:54:25,4.1888,001691626,2020-12-25 05:54:25
2,2020/12/25,05:55:25,4.1888,001691627,2020-12-25 05:55:25
3,2020/12/25,05:56:28,4.1888,001691628,2020-12-25 05:56:28
4,2020/12/25,05:57:28,4.1888,001691629,2020-12-25 05:57:28


In [16]:
from allsky.analysis import quick_exposure_linregress
quick_exposure_linregress(ideal_df[ideal_df["exposure"]<50.0])

array([ 2.22281202e-03, -1.62486599e+03])